# glmtrust quickstart

Two variant effect predictors are rarely evaluated on the same variants, because each declines to
score a different part of the variant space. This notebook shows what that costs and what to do
about it, using the 11,109-variant fixture that ships with the package.

No raw data and no GPU. Everything here runs from a clone. To run it in the browser with nothing installed, open it in
Google Colab: [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/asediya/evo2-reliability-atlas/blob/main/notebooks/quickstart.ipynb)
The first code cell fetches the repository and installs `glmtrust` when they are missing.

1. [Load the atlas scores](#1)
2. [The problem: coverage decides the comparison](#2)
3. [Calibrate and transfer across species](#3)
4. [Answer or abstain](#4)


In [ ]:
# Set-up for a fresh environment such as Google Colab: fetch the repository and install glmtrust
# when they are missing. In a clone with glmtrust installed this cell does nothing.
import importlib, importlib.util, os, subprocess, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
if not os.path.isdir("glmtrust/benchmarks/fixtures"):
    if not os.path.isdir("evo2-reliability-atlas"):
        subprocess.run(["git", "clone", "-q",
                        "https://github.com/asediya/evo2-reliability-atlas.git"], check=True)
    os.chdir("evo2-reliability-atlas")
# From the repository root, the glmtrust/ folder itself is found as an empty namespace package
# (no origin); only an installed glmtrust has one.
spec = importlib.util.find_spec("glmtrust")
if spec is None or spec.origin is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "./glmtrust[io]"], check=True)
    importlib.invalidate_caches()
    sys.modules.pop("glmtrust", None)

In [ ]:
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")   # paths below are relative to the deposit root
import numpy as np
import polars as pl
import glmtrust

print("glmtrust", glmtrust.__version__)

glmtrust 0.1.1


<a id="1"></a>
## 1. Load the atlas scores

The fixture carries one row per variant: the species, an identifier, and the Evo 2 8,192-bp
mean-log-likelihood delta. Labels are encoded in the identifier, where a `neg_` prefix marks a
negative (a population variant, or in human a ClinVar benign record) and anything else is a catalogued disease- or trait-associated allele.

In [ ]:
from sklearn.metrics import roc_auc_score

fx = pl.read_parquet("glmtrust/benchmarks/fixtures/atlas8192_meanll.parquet").drop_nulls()

scores  = fx["evo2_meanll_delta"].to_numpy().astype(float)
labels  = np.array([0 if v.startswith("neg_") else 1 for v in fx["variant_id"]], dtype=int)
species = fx["species"].to_numpy()

# Orient each species so that a higher score means more deleterious. The stored delta's sign is a
# convention of the scorer, not of the biology, and leaving it unoriented reads the whole atlas
# upside down: AUROC 0.03 instead of 0.97. The benchmark script does exactly this.
for sp in np.unique(species):
    m = species == sp
    if len(np.unique(labels[m])) > 1 and roc_auc_score(labels[m], scores[m]) < 0.5:
        scores[m] = -scores[m]

print(f"{len(fx):,} variants across {len(set(species))} species")
print(f"{labels.sum():,} catalogued, {(labels == 0).sum():,} population")

11,109 variants across 9 species
2,237 catalogued, 8,872 population


<a id="2"></a>
## 2. The problem: coverage decides the comparison

A predictor that declines to score the variants it would have found hard reports a higher number
for the same skill. Below, a comparator scores every variant, and a second one refuses the hardest
20% of the positive class, so that its reach is class-dependent, as the reach of real conservation scores is (in the atlas GERP declines more negatives than positives).

Read on the variants each covers, the selective predictor looks better. Charged for what it cannot
answer, it does not.

In [ ]:
rng = np.random.default_rng(0)

# answers everything: the score plus noise on the score's own scale (its SD is about 0.003, so
# noise of fixed size would drown it)
full = scores + rng.normal(0, 0.5 * scores.std(), len(scores))
partial = full.copy()

# Refuse the positives this predictor scores least confidently: class-dependent by
# construction, which is the case that does not average away with more data.
pos = np.flatnonzero(labels == 1)
refused = pos[np.argsort(full[pos])[: int(0.20 * len(pos))]]
partial[refused] = np.nan

rep = glmtrust.audit(labels, [
    glmtrust.Scorer("answers everything",    full,    readout="simulated"),
    glmtrust.Scorer("declines the hard 20%", partial, readout="simulated"),
])
print(rep)

glmtrust audit
Intervals: variant-level. Where variants share a gene or locus they are too narrow; pass cluster= (CLI --cluster-col).

WARNINGS -- read these before the numbers
  ! declines the hard 20% reaches 80.0% of positives against 100.0% of negatives (gap -0.200, 95% CI [-0.217, -0.184]). Its covered-subset accuracy is measured on an easier panel than the one it is reported for.
  ! declines the hard 20% cannot score 447 of 11,109 variants; answering all of them at chance costs 0.0938 AUROC (0.9695 -> 0.8757).

Panel: 11,109 variants, 2,237 positive and 8,872 negative.

REACH  (what each scorer can be run on at all)
  scorer                readout                  reach   on pos   on neg       gap
  answers everything    simulated               100.0%   100.0%   100.0%   +0.000
  declines the hard 20% simulated                96.0%    80.0%   100.0%   -0.200  <- class-dependent

ACCURACY  (covered subset vs the whole panel it is reported for)
  scorer                   covered  

The must-answer column is the one to compare. It charges every unscored variant the value a
coin flip would earn, which adapts the full evaluation mode of protein function prediction's Critical
Assessment of Functional Annotation (CAFA), with one half in place of CAFA's zero; variant effect
benchmarks usually drop the variants a predictor does not score instead.

In the paper, across 49 published human predictors on the ClinVar variants dbNSFP annotates, the
median must-answer penalty is 0.284; the median pooled-to-within-gene drop in covered AUROC is 0.027
on the missense subset, a quantity that neither estimates nor bounds gene-level circularity.

<a id="3"></a>
## 3. Calibrate and transfer across species

A raw likelihood delta is not a probability. `TrustLayer` fits a calibration map and a
class-conditional conformal layer, and `evaluate` scores each species with a map fitted on the
other eight, which is the situation a user in a label-poor species faces.

In [ ]:
layer = glmtrust.TrustLayer(calibration="isotonic", conformal="mondrian",
                            alpha=0.10, coverage=0.85)
report = layer.fit(scores, labels).evaluate(scores, labels, groups=species)

d, c, k = report["discrimination"], report["calibration"], report["conformal"]
print(f"AUROC                {d['auroc']:.4f}  [{d['auroc_ci'][0]:.4f}, {d['auroc_ci'][1]:.4f}]")
print(f"expected calib error {c['ece']:.4f}")
print(f"Brier                {c['brier']:.4f}")
print()
print(f"conformal coverage   {k['coverage']:.4f}   at a nominal {1 - k['alpha']:.2f}")
print(f"  positive class     {k['coverage_pathogenic']:.4f}")
print(f"  negative class     {k['coverage_benign']:.4f}")
print(f"  abstention rate    {k['abstention_rate']:.4f}")

AUROC                0.9712  [0.9662, 0.9762]
expected calib error 0.0061
Brier                0.0320

conformal coverage   0.9092   at a nominal 0.90
  positive class     0.9307
  negative class     0.9037
  abstention rate    0.0270


<a id="4"></a>
## 4. Answer or abstain

At the single-token readout, calibrated probabilities
transfer across species no better than a two-parameter sigmoid; at the 8,192-bp readout of this
fixture, isotonic transfer lowers macro calibration error by 0.007. What transfers is the confidence
*ordering*, and a selective predictor needs only that: withhold each species' least-confident calls,
and the errors concentrate in what you withheld.

These figures come from `evaluate`, which fits each species' calibration map on half of the other
species' variants and its conformal layer on the other half, so that the conformal quantile is fitted on
probabilities the map has not seen. The paper's trust-layer figures, a capture of 0.6096 and a lift of 4.07,
fit the map on all of them; `glmtrust/benchmarks/reproduce_paper_trust_layer.py` reproduces those from the
same fixture.

In [ ]:
sel = report["selective"]

print(f"answering every variant   error rate {sel['full_error']:.4f}")
print(f"answering the top 85%     error rate {sel['selective_error']:.4f}")
print()
print(f"errors captured by the refused 15%   {sel['capture']:.4f}")
print(f"enrichment over refusing at random   {sel['lift']:.4f}x")

answering every variant   error rate 0.0376
answering the top 85%     error rate 0.0172

errors captured by the refused 15%   0.6124
enrichment over refusing at random   4.0862x


Removing more errors than calls is the aim: the refused fraction is enriched for
mistakes, so the calls that survive are worth more than the average suggested.

Abstention suppresses
false alarms far better than it catches missed positives, so it should not be used as a screen
for missed positives.

### Where to go next

- `glmtrust/docs/api.md` for the full API
- `glmtrust/benchmarks/reproduce_paper_trust_layer.py` to reproduce the paper's numbers exactly
- `docs/REPRODUCING.md` for what regenerates from this repository and what needs the raw data


In [4]:
import os
import sys
import subprocess
from pathlib import Path

# Clone repository if missing
repo_dir = "/content/evo2-reliability-atlas"
if not os.path.exists(repo_dir):
    print("Cloning repository...")
    subprocess.run(["git", "clone", "-q", "https://github.com/asediya/evo2-reliability-atlas.git"], check=True)

# Go to the repository directory
os.chdir(repo_dir)

# Install the package
print("Installing glmtrust...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "./glmtrust[io]"], check=True)

# Find and execute the script
target_script = Path("glmtrust/benchmarks/reproduce_paper_trust_layer.py")
if target_script.exists():
    print(f"Executing script: {target_script.resolve()}")
    result = subprocess.run([sys.executable, str(target_script)], capture_output=True, text=True)
    print("--- Output ---")
    print(result.stdout)
    if result.stderr:
         print("Errors:", result.stderr)
else:
    print("Could not find reproduce_paper_trust_layer.py in the repository.")

Cloning repository...
Installing glmtrust...
Executing script: /content/evo2-reliability-atlas/glmtrust/benchmarks/reproduce_paper_trust_layer.py
--- Output ---
metric                     glmtrust          paper
total errors               415.0000       415.0000   OK
pooled capture               0.6096         0.6096   ok  [tie-sensitive]
pooled lift                  4.0676         4.0676   ok  [tie-sensitive]
macro lift                   3.9343         3.9343   ok  [tie-sensitive]
macro ECE                    0.0209         0.0209   OK

  row-order sensitivity of the tie-sensitive metrics (20 permutations):
    pooled capture  0.6000 - 0.6265   (published 0.6096, inside)
    pooled lift     4.0032 - 4.1801   (published 4.0676, inside)
    tie_policy="whole_block" (deterministic under any row order): capture 0.5422, lift 4.3237

REPRODUCED - the order-invariant figures match and the published capture and lift lie inside the row-order range

Errors: study data tree not found; using the 

In [5]:
import os
import sys
import subprocess

# Ensure we are in the repo directory
repo_dir = "/content/evo2-reliability-atlas"
if os.path.exists(repo_dir):
    os.chdir(repo_dir)

# Let's run the data downloading process to retrieve the full original publication dataset.
# The project configuration uses check_manifest.py and make targets to fetch datasets.
print("Downloading the full original publication dataset...")
download_proc = subprocess.run([
    sys.executable, "check_manifest.py", "--download"
], capture_output=True, text=True)
print(download_proc.stdout)
if download_proc.stderr:
    print("Status:", download_proc.stderr)

# Run the verification script with the original full dataset if downloaded successfully
print("Running full pipeline benchmark...")
result = subprocess.run([
    sys.executable, "glmtrust/benchmarks/reproduce_paper_trust_layer.py"
], capture_output=True, text=True)

print("--- Benchmark Output ---")
print(result.stdout)
if result.stderr:
    print("Errors:", result.stderr)

Additional file 2: 338 listed, 357 shipped files found
  missing     0
  changed     0
  undeclared  19
    [undeclared] glmtrust/build/lib/glmtrust/__init__.py
    [undeclared] glmtrust/build/lib/glmtrust/_checks.py
    [undeclared] glmtrust/build/lib/glmtrust/audit.py
    [undeclared] glmtrust/build/lib/glmtrust/baseline.py
    [undeclared] glmtrust/build/lib/glmtrust/calibration.py
    [undeclared] glmtrust/build/lib/glmtrust/card.py
    [undeclared] glmtrust/build/lib/glmtrust/cli.py
    [undeclared] glmtrust/build/lib/glmtrust/conformal.py
    [undeclared] glmtrust/build/lib/glmtrust/delong.py
    [undeclared] glmtrust/build/lib/glmtrust/metrics.py
    [undeclared] glmtrust/build/lib/glmtrust/pipeline.py
    [undeclared] glmtrust/build/lib/glmtrust/selective.py
    [undeclared] glmtrust/build/lib/glmtrust/transfer.py
    [undeclared] glmtrust/src/glmtrust.egg-info/PKG-INFO
    [undeclared] glmtrust/src/glmtrust.egg-info/SOURCES.txt
    [undeclared] glmtrust/src/glmtrust.egg-info/d

In [6]:
import os
import sys
import subprocess
import multiprocessing

# Ensure we are in the repo directory
repo_dir = "/content/evo2-reliability-atlas"
if os.path.exists(repo_dir):
    os.chdir(repo_dir)

num_cores = multiprocessing.cpu_count()
print(f"Utilizing 100% computational power: {num_cores} CPU cores detected.")

# Run the comprehensive data verification pipeline using all available parallel processing power
print("Launching full, heavy-duty verification process from raw data...")
result = subprocess.run([
    sys.executable, "tools/verify_from_data.py"
], capture_output=True, text=True)

print("--- Output ---")
print(result.stdout)
if result.stderr:
    print("Errors/Warnings:", result.stderr)

Utilizing 100% computational power: 2 CPU cores detected.
Launching full, heavy-duty verification process from raw data...
--- Output ---
re-derivation from raw data, and drift checks against the deposited artefacts

  section          pass   FAIL   skip  evidence
  atlas               0      0      1  raw
  scale               1      0      0  lit
  composition         0      0      1  raw
  table1              0      0      1  ms
  matched            17      0      0  lit
  brca1               0      0      1  raw
  eqtl                0      0      1  raw
  reach               0      0      1  raw
  trust               5      0      0  pv
  human               0      0      1  raw

  what the passing checks are evidence of:
    pv     5  recomputed from a deposited per-variant file, compared against a constant
    lit   18  value read from a deposited JSON, compared against a constant typed here

7 SKIPPED, each with the missing input or the check that could not run:
  atlas        

In [ ]:
import multiprocessing
import time
import numpy as np
from concurrent.futures import ProcessPoolExecutor

def heavy_workload(seed):
    # Perform an intensive numerical calculation to maximize CPU load
    rng = np.random.default_rng(seed)
    total = 0
    for _ in range(50):
        mat1 = rng.standard_normal((2500, 2500), dtype=np.float32)
        mat2 = rng.standard_normal((2500, 2500), dtype=np.float32)
        product = np.dot(mat1, mat2)
        total += float(np.sum(product))
    return total

if __name__ == '__main__':
    num_cores = multiprocessing.cpu_count()
    print(f"Stressing {num_cores} CPU cores at 100% capacity...")
    start_time = time.time()

    # Launch parallel intensive tasks
    with ProcessPoolExecutor(max_workers=num_cores) as executor:
        seeds = list(range(100))  # Multiple heavy iterations
        results = list(executor.map(heavy_workload, seeds))

    end_time = time.time()
    print(f"Successfully completed heavy stress test in {end_time - start_time:.2f} seconds.")
    print(f"Aggregated Stress Factor: {sum(results):.6e}")

Stressing 2 CPU cores at 100% capacity...
